# Iteration 2 – Time-to-close by district (Shenoy)

Task file: `docs/tasks/iter2-shenoy.md` · Reviewer: Ma'el · Outputs: `outputs/iter2/time_to_close/`

**Question:** do districts differ in how fast GlendaleOne requests close once request type is accounted for, and how much of the raw difference is just type mix?

**Grain:** one row = one GlendaleOne request. Input: `data/cleaned/glendaleone_clean.csv` (extract loaded 2026-08-06), downloaded from the team Drive.

**Wording rule:** "slower than typical for its type", never "late" or "missed target". The City's escalation targets are not downloaded (DECISIONS 2026-09-25).

Run with **Kernel → Restart & Run All**. Before committing, clear outputs:
`jupyter nbconvert --clear-output --inplace notebooks/iter2_time_to_close_shenoy.ipynb`

## 0. Setup
Standard load block, verbatim from `docs/ANALYSIS_STANDARDS.md` section 2.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "cleaned"
DISTRICTS = ["BARREL", "CACTUS", "CHOLLA", "OCOTILLO", "SAHUARO", "YUCCA"]
START, END = "2020-01-01", "2026-07-31"      # 79 full months
YEARS = 79 / 12

req = pd.read_csv(DATA / "glendaleone_clean.csv",
                  parse_dates=["request_date", "close_date"], low_memory=False)
req["district"] = req["district"].astype("string").str.strip().str.upper()
n_raw = len(req)
req_w = req[req["request_date"].between(START, END)]
n_window = len(req_w)
req_w = req_w[req_w["district"].isin(DISTRICTS)].copy()
n_final = len(req_w)
print(n_raw, n_window, n_final, n_window - n_final)   # last number = rows without a district

In [ ]:
import numpy as np
import textwrap
from scipy import stats
from statsmodels.stats.multitest import multipletests
import seaborn as sns, matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")        # seaborn 0.13.2
BLUE = "#2a78d6"                                        # default single hue
DISTRICT_COLORS = {"BARREL": "#2a78d6", "CACTUS": "#eb6834", "CHOLLA": "#1baf7a",
                   "OCOTILLO": "#eda100", "SAHUARO": "#e87ba4", "YUCCA": "#008300"}
GREY = "#a3adb8"
LIGHT = "#9cc3ef"

OUT = ROOT / "outputs" / "iter2" / "time_to_close"
OUT.mkdir(parents=True, exist_ok=True)

TYPE_COL = "request_type"          # 155 exact types: the main type control (see the note in section 2)
GROUP_COL = "request_type_group"   # 17 groups: the ANALYSIS_STANDARDS control, kept as the comparison
TOP_K = 8                         # task 2: the 8 biggest types
SMALL_CELL = 30                   # district x type cells under this n get a * (flagged, never dropped)
TRASH = "Trash/Recycle Services"
SOURCE = "Source: City of Glendale GlendaleOne External Requests (extract loaded 2026-08-06)."
WINDOW_TXT = "closed requests opened 2020-01 to 2026-07"

def fmt_p(p):
    if pd.isna(p):
        return "n/a"
    if p < 0.001:
        return "p < .001"
    s = f"{p:.3f}"
    return "p = " + (s[1:] if s.startswith("0") else s)

def eps2_label(e):   # rule of thumb for epsilon squared (Tomczak & Tomczak 2014)
    return "negligible" if e < 0.01 else "small" if e < 0.08 else "medium" if e < 0.26 else "large"

def v_label(v):      # rule of thumb for Cramér's V with df = 1 (Cohen 1988)
    return "negligible" if v < 0.1 else "small" if v < 0.3 else "medium" if v < 0.5 else "large"

def nice(d):         # BARREL -> Barrel, for chart and slide text
    return str(d).title()

SHORT = {"Trash/Recycle Services": "Trash/Recycle", "Streets/Sidewalks/Medians/Corners": "Streets & Sidewalks",
         "Traffic Signals/Signs": "Signals & Signs", "Water/Sewer Services": "Water/Sewer",
         "Park & Recreation Facilities": "Parks & Rec", "Zoning, Permitting, & Inspections": "Zoning & Permits",
         "Public Transportation": "Public Transit", "Americans with Disabilities Act (ADA)": "ADA",
         "Library Services": "Library", "Miscellaneous": "Misc.",
         "Residential - Container Damaged": "Container damaged", "Abandoned Vehicle - Roadway": "Abandoned vehicle",
         "Street Light or Pedestrian Light Issue": "Street light", "Traffic Sign - Repair": "Traffic sign repair",
         "Trash/Debris in Residential Areas": "Trash & debris", "Parking Enforcement": "Parking enforcement",
         "Residential - Missed Regular Collection": "Missed collection",
         "Residential - Container Missing": "Container missing", "Bulk Trash - Other": "Bulk trash",
         "Roadside, Alley, or Median Landscape": "Roadside landscape",
         "Street or Sidewalk - Damage": "Street/sidewalk damage",
         "Rent Assistance (Rent, Mortgage, Eviction)": "Rent assistance",
         "Traffic Enforcement - Public Property": "Traffic enforcement",
         "Commercial - Missed Regular Collection": "Commercial missed collection"}
def short(t):        # short type-group name for charts and talking points
    return SHORT.get(str(t), str(t))

def days_txt(v):
    return f"{v:g} day" if v == 1 else f"{v:g} days"

def who(series, value):   # every district tied at a value, e.g. "Barrel and Cactus"
    names = [nice(d) for d in series.index[series == value]]
    return names[0] if len(names) == 1 else ", ".join(names[:-1]) + " and " + names[-1]

odd = []             # anything that looks wrong, for the chat post
def check(name, ok, detail=""):
    print(("PASS  " if ok else "FLAG  ") + name + (f"  ({detail})" if detail else ""))
    if not ok:
        odd.append(f"{name}: {detail}" if detail else name)

## 1. Data check (task 1)

Does `glendaleone_clean.csv` look right: dates, status, close dates? Every FLAG line goes into the chat post at the end. Checks follow `docs/ANALYSIS_STANDARDS.md` section 5.

In [ ]:
# Flags as real booleans, whether pandas read them as bool or as text
def to_bool(s):
    return s.astype("string").str.strip().str.lower().eq("true").fillna(False).astype(bool)

for df in (req, req_w):
    for c in [c for c in df.columns if c.startswith("flag_")]:
        df[c] = to_bool(df[c])

print("Grain: one row = one GlendaleOne request\n")
print(req_w[["request_number", "status", "request_date", "close_date", "district", GROUP_COL, TYPE_COL]].dtypes, "\n")

# Expected counts from ANALYSIS_STANDARDS section 2
check("n_raw is 106,795 (the Drive copy)", n_raw == 106_795, f"n_raw = {n_raw:,}")
n_before, n_after = int((req["request_date"] < START).sum()), int((req["request_date"] > END).sum())
n_no_date = int(req["request_date"].isna().sum())
print(f"Outside the window: {n_before:,} before 2020-01-01, {n_after:,} after 2026-07-31, {n_no_date:,} with no request_date")
n_no_district = n_window - n_final
check("about 2.2% of the window has no district", 0.015 <= n_no_district / n_window <= 0.03,
      f"{n_no_district:,} rows = {n_no_district / n_window:.2%}")

# Missing values, treating NULL, '' and 'N/A' alike
def n_missing(s):
    txt = s.astype("string").str.strip()
    return int((s.isna() | txt.isin(["", "N/A"]).fillna(False)).sum())

use = ["request_number", "status", "request_date", "close_date", "district", GROUP_COL, TYPE_COL]
display(pd.Series({c: n_missing(req_w[c]) for c in use}, name="missing_rows_in_analysis_set").to_frame())

check("request_number is unique", req_w["request_number"].is_unique)
check("exactly the 6 district names", set(req_w["district"].unique()) == set(DISTRICTS),
      str(sorted(req_w["district"].unique())))
check("request_date inside the window", req_w["request_date"].between(START, END).all())

# Status and close dates
status = req_w["status"].value_counts()
display(status.to_frame("rows"))
share_closed = status.get("Closed", 0) / n_final
check("about 99.6% of requests are Closed", share_closed > 0.99, f"{share_closed:.2%} closed")
n_closed_no_date = int((req_w["status"].eq("Closed") & req_w["close_date"].isna()).sum())
check("every Closed request has a close_date", n_closed_no_date == 0, f"{n_closed_no_date:,} Closed rows without one")
n_open_with_date = int((req_w["status"].ne("Closed") & req_w["close_date"].notna()).sum())
if n_open_with_date:
    odd.append(f"{n_open_with_date:,} requests are On Hold / In Progress / Open but carry a close_date "
               "(reopened?); the Closed filter drops them")
n_flag = int(req_w["flag_close_before_request"].sum())
print(f"Rows closing before they were opened (flag_close_before_request): {n_flag} – dropped below, not fixed")

loaded = pd.to_datetime(req["date_loaded"], errors="coerce").max() if "date_loaded" in req else pd.NaT
close_max = req_w["close_date"].max()
print(f"Latest close_date: {close_max.date()} · latest date_loaded: {loaded}")
if pd.notna(loaded):
    check("no close_date after the extract was loaded", close_max <= loaded.normalize(),
          f"latest close_date {close_max.date()}")

# Time coverage: 79 months, none empty
months = req_w["request_date"].dt.to_period("M").value_counts().sort_index()
full = pd.period_range("2020-01", "2026-07", freq="M")
check("all 79 months have requests", months.reindex(full).notna().all() and len(full) == 79,
      f"{months.reindex(full).isna().sum()} empty months")
print(f"Requests per month: min {months.min():,} ({months.idxmin()}), max {months.max():,} ({months.idxmax()})")

# Right-censoring: recent requests that were still open on the extract date are missing from closed-only timing
recent = req_w[req_w["request_date"] >= "2026-05-01"]
by_m = recent.groupby(recent["request_date"].dt.to_period("M"))["status"].apply(lambda s: s.eq("Closed").mean())
print("Share closed, last three months:", ", ".join(f"{k}: {v:.1%}" for k, v in by_m.items()))
if len(by_m) and by_m.min() < 0.98:
    odd.append(f"only {by_m.min():.1%} of {by_m.idxmin()} requests were closed by the extract date, "
               "so the most recent months lean fast in closed-only timing")

### Filters and the filter log
Closed requests only, window 2020-01 to 2026-07, the 6 districts, and drop the one request that closes before it was opened (don't fix it). `days_to_close = (close_date - request_date).dt.days`; dates are date-only UTC, so a same-day close is 0.

In [ ]:
steps = [("raw file", n_raw),
         ("request_date 2020-01-01 to 2026-07-31", n_window),
         ("district is one of the 6", n_final)]
closed = req_w[req_w["status"].eq("Closed")]
steps.append(("status == Closed", len(closed)))
closed = closed[closed["close_date"].notna()]
steps.append(("close_date not null", len(closed)))
closed = closed[~closed["flag_close_before_request"]]
steps.append(("drop flag_close_before_request (not fixed)", len(closed)))

ttc = closed.copy()
ttc["days_to_close"] = (ttc["close_date"] - ttc["request_date"]).dt.days
n_closed = len(ttc)

flog = pd.DataFrame(steps, columns=["step", "rows_after"])
flog["rows_removed"] = (-flog["rows_after"].diff()).fillna(0).astype(int)
assert flog["rows_removed"].sum() == n_raw - n_closed       # excluded counts add up
flog.to_csv(OUT / "filter_log.csv", index=False)
display(flog)

check("days_to_close is at least 0 on every row", (ttc["days_to_close"] >= 0).all(),
      f"min {ttc['days_to_close'].min()}")

### Table 1 – days to close by district (closed requests)

In [ ]:
def summarise(df, by):
    g = df.groupby(by)["days_to_close"]
    t = pd.DataFrame({
        "n": g.size(),
        "mean_days": g.mean(),
        "median_days": g.median(),
        "p25_days": g.quantile(0.25),
        "p75_days": g.quantile(0.75),
        "p90_days": g.quantile(0.90),
        "max_days": g.max(),
    })
    flags = df.assign(_same=df["days_to_close"].eq(0), _over=df["days_to_close"].gt(365))
    t["share_same_day"] = flags.groupby(by)["_same"].mean()
    t["share_over_365"] = flags.groupby(by)["_over"].mean()
    return t

t1 = summarise(ttc, "district").reindex(DISTRICTS)
t1.index.name = "district"
t1 = t1.reset_index()
city = summarise(ttc.assign(_all="ALL"), "_all").iloc[0]

# Reconcile and check two ways
check("district n sums to n_closed", t1["n"].sum() == n_closed)
vc = ttc["district"].value_counts().reindex(DISTRICTS).to_numpy()
check("n by value_counts matches groupby", (vc == t1["n"].to_numpy()).all())
med2 = np.array([np.median(ttc.loc[ttc["district"] == d, "days_to_close"]) for d in DISTRICTS])
check("median by numpy matches groupby", np.allclose(med2, t1["median_days"]))

t1.round({"mean_days": 2, "median_days": 1, "p25_days": 1, "p75_days": 1, "p90_days": 1,
          "share_same_day": 4, "share_over_365": 4}).to_csv(OUT / "days_to_close_by_district.csv", index=False)

show = t1.assign(same_day=(100 * t1["share_same_day"]).round(1)).loc[
    :, ["district", "n", "mean_days", "median_days", "p90_days", "same_day"]].round(1)
print(f"Citywide: n = {n_closed:,}, mean {city['mean_days']:.1f}, median {city['median_days']:g}, "
      f"p90 {city['p90_days']:g}, same day {city['share_same_day']:.1%}")
display(show.rename(columns={"same_day": "same_day_pct"}))

### Outliers (flagged, not deleted)
IQR fence and anything over 365 days. One result shown with and without the over-365 requests.

In [ ]:
q1, q3 = ttc["days_to_close"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
keep = ttc["days_to_close"] <= 365
outl = pd.DataFrame({
    "n": ttc.groupby("district").size(),
    "n_over_iqr_fence": ttc[ttc["days_to_close"] > fence].groupby("district").size(),
    "n_over_365": ttc[~keep].groupby("district").size(),
    "median_days_all": ttc.groupby("district")["days_to_close"].median(),
    "median_days_excl_over_365": ttc[keep].groupby("district")["days_to_close"].median(),
    "mean_days_all": ttc.groupby("district")["days_to_close"].mean(),
    "mean_days_excl_over_365": ttc[keep].groupby("district")["days_to_close"].mean(),
}).reindex(DISTRICTS).fillna({"n_over_iqr_fence": 0, "n_over_365": 0})
outl.index.name = "district"
outl = outl.reset_index().astype({"n_over_iqr_fence": int, "n_over_365": int}).round(2)
outl.to_csv(OUT / "outlier_check.csv", index=False)
n_over_365 = int((~keep).sum())
print(f"IQR fence: {fence:g} days. Over the fence: {(ttc['days_to_close'] > fence).sum():,}. "
      f"Over 365 days: {n_over_365:,} ({n_over_365 / n_closed:.2%}).")
display(outl)

## 2. Type-corrected comparison (task 2)

**Definition.** A closed request is **slower than typical for its type** when its `days_to_close` is above the citywide median for its exact `request_type` (155 types), computed on this same analysis set. Ties count as not slower. Dates are whole days, so many requests close exactly at their type's median, which puts the citywide share below 50%. The reference line is that citywide share, not 50%.

**Tests, written before the first run** (ANALYSIS_STANDARDS section 6):

1. Kruskal-Wallis on raw `days_to_close` across the 6 districts; effect size epsilon squared = H / (N - 1).
2. Chi-square on district x slower than typical (6 x 2); effect size Cramér's V, plus Pearson residuals per district.
3. The same chi-square with the citywide median of all requests as the yardstick ("raw"). The gap between the highest and lowest district share, raw vs type-corrected, shows how much of the raw gap lines up with type mix.

Exploratory (labelled as such): Kruskal-Wallis within each of the 8 most common types, Holm-corrected.

**Change after the first run.** The first version used the 17 `request_type_group` values as the type control (ANALYSIS_STANDARDS section 3), with the exact types as a sensitivity check. The two disagreed sharply: with exact types the district gap left after correction was about a quarter of the size, and a different district came out slowest. Section 2b shows why: one group can hold types with very different typical times (in Trash/Recycle Services, container repairs take several days while trash and debris pickups close the same day), and districts file them in different proportions. The exact type also matches the task file ("the median for their own request type"), so it is now the main control. The group-level version stays in the tables and in `stat_tests.csv` as the comparison, and `findings.md` records the change.

In [ ]:
type_med = ttc.groupby(TYPE_COL)["days_to_close"].median()
group_med = ttc.groupby(GROUP_COL)["days_to_close"].median()
ttc["type_median_days"] = ttc[TYPE_COL].map(type_med)
ttc["group_median_days"] = ttc[GROUP_COL].map(group_med)
city_median = ttc["days_to_close"].median()
ttc["slower_than_type"] = ttc["days_to_close"] > ttc["type_median_days"]     # main measure: exact type
ttc["slower_than_group"] = ttc["days_to_close"] > ttc["group_median_days"]   # comparison: 17 groups
ttc["slower_than_city"] = ttc["days_to_close"] > city_median                 # raw yardstick

city_share_type = ttc["slower_than_type"].mean()
city_share_group = ttc["slower_than_group"].mean()
city_share_raw = ttc["slower_than_city"].mean()

g = ttc.groupby("district")
st = pd.DataFrame({
    "n": g.size(),
    "n_slower_than_type": g["slower_than_type"].sum(),
    "share_slower_than_type": g["slower_than_type"].mean(),
    "share_slower_than_type_group": g["slower_than_group"].mean(),
    "share_slower_than_city_median": g["slower_than_city"].mean(),
    "share_trash_recycle": g[GROUP_COL].apply(lambda s: s.eq(TRASH).mean()),
}).reindex(DISTRICTS)
st["pp_vs_citywide_type"] = 100 * (st["share_slower_than_type"] - city_share_type)
st.index.name = "district"
st = st.reset_index()

def gap(col):        # highest minus lowest district share
    return st[col].max() - st[col].min()

spread_raw = gap("share_slower_than_city_median")
spread_group = gap("share_slower_than_type_group")
spread_type = gap("share_slower_than_type")
shrink = 1 - spread_type / spread_raw if spread_raw > 0 else np.nan
shrink_group = 1 - spread_group / spread_raw if spread_raw > 0 else np.nan
# How the gap changes once type is accounted for (used in chart 2, findings and talking points)
MIX = "shrinks" if shrink >= 0.10 else "widens" if shrink <= -0.10 else "similar"

check("type-corrected n sums to n_closed", st["n"].sum() == n_closed)
two_way = ttc.loc[ttc["slower_than_type"], "district"].value_counts().reindex(DISTRICTS).fillna(0).to_numpy()
check("slower counts by value_counts match groupby", (two_way == st["n_slower_than_type"].to_numpy()).all())

st.round({"share_slower_than_type": 4, "share_slower_than_type_group": 4, "share_slower_than_city_median": 4,
          "share_trash_recycle": 4, "pp_vs_citywide_type": 2}).to_csv(OUT / "type_corrected_share_by_district.csv",
                                                                       index=False)

tm = (ttc.groupby([TYPE_COL, GROUP_COL])["days_to_close"].agg(n="size", median_days="median")
         .reset_index().sort_values("n", ascending=False))
tm.to_csv(OUT / "type_medians_citywide.csv", index=False)
small_types = tm.loc[tm["n"] < SMALL_CELL, TYPE_COL]
n_small_rows = int(ttc[TYPE_COL].isin(small_types).sum())

print(f"Citywide median {days_txt(city_median)}. Share slower than typical for its type: {city_share_type:.1%} "
      f"(type groups: {city_share_group:.1%}; raw yardstick: {city_share_raw:.1%}).")
print(f"District gap: raw {100 * spread_raw:.1f} pts -> type groups {100 * spread_group:.1f} pts -> "
      f"exact types {100 * spread_type:.1f} pts ({shrink:.0%} smaller than raw).")
print(f"{len(small_types)} types have fewer than {SMALL_CELL} closed requests ({n_small_rows:,} rows, "
      f"{n_small_rows / n_closed:.2%}).")
display(st.round(3))
display(tm.head(TOP_K))

### 2b. Why the two type controls disagree: the mix inside Trash/Recycle Services

In [ ]:
tr = ttc[ttc[GROUP_COL] == TRASH]
tr_types = tr[TYPE_COL].value_counts()
TOP_TR = tr_types.index[0]                     # the most common exact type inside Trash/Recycle
mix = pd.crosstab(tr["district"], tr[TYPE_COL]).reindex(DISTRICTS)
share_mix = mix.div(mix.sum(axis=1), axis=0)
mix_long = (share_mix.stack().rename("share_of_district_trash_recycle").reset_index()
            .merge(mix.stack().rename("n").reset_index(), on=["district", TYPE_COL]))
mix_long = mix_long[mix_long[TYPE_COL].isin(tr_types.index[:5])].copy()
mix_long["citywide_type_median_days"] = mix_long[TYPE_COL].map(type_med)
mix_long.round({"share_of_district_trash_recycle": 4}).to_csv(OUT / "trash_recycle_mix_by_district.csv", index=False)

top_share = share_mix[TOP_TR]
top_med_d = tr[tr[TYPE_COL] == TOP_TR].groupby("district")["days_to_close"].median().reindex(DISTRICTS)
hi_mix, lo_mix = top_share.idxmax(), top_share.idxmin()
print(f"{TOP_TR}: citywide median {days_txt(type_med[TOP_TR])}, {tr_types.iloc[0]:,} requests.")
print(f"It is {top_share[hi_mix]:.0%} of {nice(hi_mix)}'s {TRASH} requests but {top_share[lo_mix]:.0%} of "
      f"{nice(lo_mix)}'s. Within it, district medians run from {top_med_d.min():g} to {days_txt(top_med_d.max())}.")
display((100 * share_mix[tr_types.index[:5]]).round(1))
display(type_med[tr_types.index[:5]].to_frame("citywide_median_days"))

### District x request-type table – median days, the 8 most common types
The same table at the 17-group level is saved as `median_days_by_district_and_type_group.csv` for comparison.

In [ ]:
def district_by(level):
    top = ttc[level].value_counts().head(TOP_K).index.tolist()
    sub = ttc[ttc[level].isin(top)]
    t = (sub.groupby(["district", level])["days_to_close"]
            .agg(n="size", median_days="median", mean_days="mean", p90_days=lambda s: s.quantile(0.9))
            .reset_index())
    t["citywide_median_days"] = t[level].map(sub.groupby(level)["days_to_close"].median())
    t["small_cell"] = t["n"] < SMALL_CELL
    t["district"] = pd.Categorical(t["district"], DISTRICTS, ordered=True)
    t[level] = pd.Categorical(t[level], top, ordered=True)
    return top, t.sort_values(["district", level]).reset_index(drop=True)

top, dt = district_by(TYPE_COL)
top_g, dt_g = district_by(GROUP_COL)
dt.round({"mean_days": 2, "p90_days": 1}).to_csv(OUT / "median_days_by_district_and_type.csv", index=False)
dt_g.round({"mean_days": 2, "p90_days": 1}).to_csv(OUT / "median_days_by_district_and_type_group.csv", index=False)
share_top8 = ttc[TYPE_COL].isin(top).mean()

wide = dt.pivot(index="district", columns=TYPE_COL, values="median_days").reindex(index=DISTRICTS, columns=top)
wide_n = dt.pivot(index="district", columns=TYPE_COL, values="n").reindex(index=DISTRICTS, columns=top)
within = pd.DataFrame({"low": wide.min(), "high": wide.max()})
within["range_days"] = within["high"] - within["low"]
print(f"The {TOP_K} most common types hold {share_top8:.0%} of closed requests.")
display(wide)
display(within.sort_values("range_days", ascending=False, kind="stable"))

### Tests

In [ ]:
rows = []

def chi_test(flag, scope, plan):
    ct = pd.crosstab(ttc["district"], ttc[flag]).reindex(DISTRICTS)
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (n_closed * (min(ct.shape) - 1)))
    rows.append(dict(test="Chi-square", scope=scope, plan=plan, n=n_closed, statistic=chi2, df=dof,
                     p_value=p, effect_size_name="cramers_v", effect_size=v))
    return ct, exp, chi2, p, v

# 1. Kruskal-Wallis on raw days across districts
groups = [ttc.loc[ttc["district"] == d, "days_to_close"].to_numpy() for d in DISTRICTS]
H, p_kw = stats.kruskal(*groups)
eps2 = H / (n_closed - 1)
rows.append(dict(test="Kruskal-Wallis", scope="days_to_close by district (raw)", plan="planned",
                 n=n_closed, statistic=H, df=len(DISTRICTS) - 1, p_value=p_kw,
                 effect_size_name="epsilon_squared", effect_size=eps2))

# 2. Chi-square: slower than typical for its exact type (main), and for its type group (original plan)
ct, exp, chi2, p_chi, V = chi_test("slower_than_type", "slower than typical for its exact type, by district",
                                   "planned; type control revised after first run")
_, _, chi2_g, p_g, V_group = chi_test("slower_than_group", "slower than typical for its type group, by district",
                                      "planned (original type control)")
# 3. Comparison: the citywide median as the yardstick
_, _, chi2_r, p_r, V_raw = chi_test("slower_than_city", "slower than citywide median (raw yardstick), by district",
                                    "planned comparison")

# Pearson residuals for the "slower" column: positive = more slower-than-typical requests than expected
exp_df = pd.DataFrame(exp, index=ct.index, columns=ct.columns)
resid = pd.DataFrame({
    "n": ct.sum(axis=1),
    "observed_slower": ct[True],
    "expected_slower": exp_df[True].round(1),
    "pearson_residual": ((ct[True] - exp_df[True]) / np.sqrt(exp_df[True])).round(2),
})
resid.index.name = "district"
resid.reset_index().to_csv(OUT / "chi2_residuals_by_district.csv", index=False)

# Exploratory: Kruskal-Wallis within each of the 8 most common types, Holm-corrected
expl = []
for t in top:
    s = ttc[ttc[TYPE_COL] == t]
    gs = [s.loc[s["district"] == d, "days_to_close"].to_numpy() for d in DISTRICTS]
    gs = [x for x in gs if len(x)]
    try:
        h, p = stats.kruskal(*gs)
    except ValueError:            # every value identical, or fewer than two groups
        h, p = np.nan, np.nan
    expl.append(dict(test="Kruskal-Wallis", scope=f"days_to_close by district within {t}", plan="exploratory",
                     n=len(s), statistic=h, df=len(gs) - 1, p_value=p,
                     effect_size_name="epsilon_squared", effect_size=h / (len(s) - 1) if len(s) > 1 else np.nan))
ps = np.array([r["p_value"] for r in expl], dtype=float)
ok = ~np.isnan(ps)
holm = np.full(len(ps), np.nan)
if ok.any():
    holm[ok] = multipletests(ps[ok], method="holm")[1]
for r, ph in zip(expl, holm):
    r["p_holm"] = ph
rows += expl

tests = pd.DataFrame(rows)
tests["p_text"] = tests["p_value"].map(fmt_p)
tests["p_holm_text"] = tests["p_holm"].map(lambda x: fmt_p(x) if pd.notna(x) else "")
tests.round({"statistic": 2, "effect_size": 4}).to_csv(OUT / "stat_tests.csv", index=False)
display(tests[["plan", "scope", "n", "statistic", "df", "p_text", "p_holm_text", "effect_size_name", "effect_size"]].round(4))
display(resid)

## 3. Charts
Chart standard: `docs/ANALYSIS_STANDARDS.md` section 7. Titles are built from the numbers, so they always match the data; reread each one before it goes on a slide.

In [ ]:
def finish(fig, ax, title, subtitle, name, note=None):
    sub = "\n".join(textwrap.fill(line, 108, break_on_hyphens=False) for line in subtitle.split("\n"))
    ax.set_title(sub, loc="left", fontsize=12, color="#444", pad=10)
    ax.annotate(textwrap.fill(title, 62), xy=(0, 1), xycoords="axes fraction",
                xytext=(0, 18 + 16 * (sub.count("\n") + 1)), textcoords="offset points",
                ha="left", va="bottom", fontsize=19, fontweight="bold")
    foot = SOURCE + (f"  {note}" if note else "")
    fig.text(ax.get_position().x0, -0.02, foot, ha="left", fontsize=10, color="#555")
    fig.savefig(OUT / name, dpi=200, bbox_inches="tight")
    plt.show()

# Chart 1: type-corrected share by district
c1 = st.assign(share_pct=100 * st["share_slower_than_type"], label=st["district"].map(nice)) \
       .sort_values("share_pct", ascending=False).reset_index(drop=True)
hi, lo = c1.iloc[0], c1.iloc[-1]
if 100 * spread_type < 5:
    title1 = (f"Once each request is compared with its own type, all six districts are within "
              f"{100 * spread_type:.1f} points of each other")
else:
    title1 = (f"{hi['label']} has the most requests slower than typical for their type: "
              f"{hi['share_pct']:.0f}% vs {lo['share_pct']:.0f}% in {lo['label']}")
fig, ax = plt.subplots(figsize=(12, 6.75))
sns.barplot(data=c1, x="label", y="share_pct", color=BLUE, ax=ax)   # one hue: district is on the axis
ax.axhline(100 * city_share_type, color="#333", lw=1.6, ls="--")
ax.text(len(c1) - 0.45, 100 * city_share_type + 0.8, f"Citywide {100 * city_share_type:.1f}%",
        ha="right", va="bottom", fontsize=13, color="#333",
        bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
for i in (0, len(c1) - 1):
    v = c1.loc[i, "share_pct"]
    ax.text(i, v / 2, f"{v:.1f}%", ha="center", va="center", fontsize=15, fontweight="bold", color="white")
ax.set_ylim(0, max(c1["share_pct"].max(), 100 * city_share_type) * 1.2)
ax.set_xlabel("")
ax.set_ylabel("Slower than typical for its type (%)")
finish(fig, ax, title1,
       f"Share of each district's closed requests slower than the citywide median for their exact request type"
       f"\n{WINDOW_TXT}, n = {n_closed:,}",
       "type_corrected_share_by_district.png")

In [ ]:
# Chart 2: raw vs type groups vs exact types, the "how much was type mix" picture
RAW_LBL = f"Raw: vs the citywide median ({days_txt(city_median)})"
GRP_LBL = "vs the median for its type group (17 groups)"
TYPE_LBL = "vs the median for its exact type (155 types)"
c2 = pd.concat([
    st.assign(measure=RAW_LBL, share_pct=100 * st["share_slower_than_city_median"]),
    st.assign(measure=GRP_LBL, share_pct=100 * st["share_slower_than_type_group"]),
    st.assign(measure=TYPE_LBL, share_pct=100 * st["share_slower_than_type"]),
])[["district", "measure", "share_pct"]]
c2["label"] = c2["district"].map(nice)
order = (st.assign(label=st["district"].map(nice))
           .sort_values("share_slower_than_city_median", ascending=False)["label"].tolist())
if MIX == "shrinks":
    title2 = (f"Comparing each request with its own type shrinks the district gap from "
              f"{100 * spread_raw:.1f} to {100 * spread_type:.1f} points")
elif MIX == "widens":
    title2 = (f"The district gap widens from {100 * spread_raw:.1f} to {100 * spread_type:.1f} points "
              f"once request type is accounted for")
else:
    title2 = (f"The district gap stays about the same once request type is accounted for "
              f"({100 * spread_raw:.1f} vs {100 * spread_type:.1f} points)")
fig, ax = plt.subplots(figsize=(12, 6.75))
sns.barplot(data=c2, x="label", y="share_pct", hue="measure", order=order,
            hue_order=[RAW_LBL, GRP_LBL, TYPE_LBL], palette=[GREY, LIGHT, BLUE], ax=ax)
for cont, meas in zip(ax.containers, [RAW_LBL, GRP_LBL, TYPE_LBL]):   # label the high and low bar, raw and exact only
    if meas == GRP_LBL:
        continue
    vals = c2[c2["measure"] == meas].set_index("label").loc[order, "share_pct"].to_numpy()
    labels = [f"{v:.0f}%" if v in (vals.max(), vals.min()) else "" for v in vals]
    ax.bar_label(cont, labels=labels, fontsize=12, padding=3)
ax.set_ylim(0, c2["share_pct"].max() * 1.38)
ax.set_xlabel("")
ax.set_ylabel("Share of closed requests slower (%)")
ax.legend(title=None, fontsize=11, loc="upper right", frameon=True)
finish(fig, ax, title2,
       f"Gap = highest minus lowest district: raw {100 * spread_raw:.1f} points, type groups "
       f"{100 * spread_group:.1f}, exact types {100 * spread_type:.1f}"
       f"\n{WINDOW_TXT}, n = {n_closed:,}",
       "raw_vs_type_corrected_by_district.png")

## 4. Heatmap of median days, district x request type (task 4)

In [ ]:
from matplotlib.ticker import MaxNLocator
ranked = within.sort_values("range_days", ascending=False, kind="stable")   # ties keep the bigger type first
big, big_type = ranked.iloc[0], ranked.index[0]
big_lo, big_hi = who(wide[big_type], big["low"]), who(wide[big_type], big["high"])
if big["range_days"] <= 3:
    title3 = (f"Same request type, similar speed: across the {TOP_K} most common types, district medians "
              f"are at most {days_txt(big['range_days'])} apart")
else:
    title3 = (f"Within the same request type, district medians differ most for {short(big_type)}: "
              f"{days_txt(big['low'])} in {big_lo} vs {days_txt(big['high'])} in {big_hi}")
annot = wide.copy().astype(object)
for d in DISTRICTS:
    for t in top:
        v, n = wide.loc[d, t], wide_n.loc[d, t]
        annot.loc[d, t] = "" if pd.isna(v) else f"{v:g}" + ("*" if n < SMALL_CELL else "")
fig, ax = plt.subplots(figsize=(12, 6.75))
sns.heatmap(wide.astype(float), annot=annot.to_numpy(), fmt="", cmap="Blues", vmin=0, linewidths=1,
            linecolor="white", annot_kws={"fontsize": 14}, ax=ax)
ax.set_xticklabels([textwrap.fill(short(t), 12, break_long_words=False) for t in top], rotation=0, fontsize=11)
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.set_major_locator(MaxNLocator(integer=True))
cbar.ax.tick_params(labelsize=11)
cbar.set_label("Median days to close", fontsize=12)
ax.set_yticklabels([nice(d) for d in DISTRICTS], rotation=0, fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("")
small_note = f"* fewer than {SMALL_CELL} requests in the cell." if (wide_n < SMALL_CELL).any().any() else None
finish(fig, ax, title3,
       f"Median days to close, the {TOP_K} most common request types (largest on the left; "
       f"{share_top8:.0%} of closed requests); {WINDOW_TXT}",
       "heatmap_median_days_district_by_type.png", note=small_note)

## 5. `findings.md`
Written from the numbers above. **Reread it and adjust the wording before the PR** – the template keeps to the repo rules (association only, "slower than typical for its type", en dashes), but the story is yours.

In [ ]:
r1 = t1.set_index("district")
med = r1["median_days"]
fast_txt, slow_txt = who(med, med.min()), who(med, med.max())
s_t = st.set_index("district")["share_slower_than_type"]
hi_t, lo_t = s_t.idxmax(), s_t.idxmin()
trash_share = (ttc[GROUP_COL] == TRASH).mean()

f1 = (f"**Before accounting for request type, district medians run from {days_txt(med.min())} in {fast_txt} to "
      f"{days_txt(med.max())} in {slow_txt}.** The typical closed request takes {days_txt(city['median_days'])} "
      f"citywide (mean {city['mean_days']:.1f}). Even so, district lines up with only {eps2:.1%} of the variation in "
      f"raw days (Kruskal-Wallis H = {H:,.0f}, epsilon squared = {eps2:.3f}, {eps2_label(eps2)} by rule of thumb, "
      f"{fmt_p(p_kw)}).")
nums2 = (f"Compared with the typical time for its exact request type, the share of requests slower than typical "
         f"runs from {100 * s_t[lo_t]:.1f}% in {nice(lo_t)} to {100 * s_t[hi_t]:.1f}% in {nice(hi_t)} "
         f"(citywide {100 * city_share_type:.1f}%): a {100 * spread_type:.1f}-point gap, vs {100 * spread_raw:.1f} "
         f"points when every request is compared with the citywide median. Cramér's V is {V:.3f} type-corrected "
         f"({v_label(V)}) vs {V_raw:.3f} raw. With the 17 broad type groups instead, the gap only narrows to "
         f"{100 * spread_group:.1f} points (finding 3 shows why).")
lead2 = {"shrinks": f"About {shrink:.0%} of the raw district gap lines up with type mix.",
         "widens": "The district gap widens once request type is accounted for.",
         "similar": "Type mix accounts for little of the district gap."}[MIX]
f2 = f"**{lead2}** {nums2}"
f3 = (f"**The broad groups hide the mix inside them.** {TOP_TR} (citywide median {days_txt(type_med[TOP_TR])}) "
      f"is {top_share[hi_mix]:.0%} of {nice(hi_mix)}'s {TRASH} requests but {top_share[lo_mix]:.0%} of "
      f"{nice(lo_mix)}'s, while within {TOP_TR} district medians run only from {top_med_d.min():g} to "
      f"{days_txt(top_med_d.max())}. ")
if big["range_days"] <= 1:
    f3 += (f"Across the {TOP_K} most common exact types ({share_top8:.0%} of closed requests), district medians "
           f"are at most {days_txt(big['range_days'])} apart.")
else:
    f3 += (f"Across the {TOP_K} most common exact types ({share_top8:.0%} of closed requests), the widest district "
           f"spread is in {big_type}: {days_txt(big['low'])} in {big_lo} to {days_txt(big['high'])} in {big_hi}.")

flog_md = "\n".join(f"| {r.step} | {r.rows_after:,} | {r.rows_removed:,} |" for r in flog.itertuples())
cens = (f" Only {by_m.min():.1%} of {by_m.idxmin()} requests were closed by the extract date."
        if len(by_m) and by_m.min() < 0.98 else "")

findings = f"""# Iteration 2 – Time-to-close by district

Owner: Shenoy · Reviewer: Ma'el · Notebook: `notebooks/iter2_time_to_close_shenoy.ipynb`

**Source and freshness:** `data/cleaned/glendaleone_clean.csv` (GlendaleOne External Requests, extract loaded 2026-08-06, team Drive copy). Window: requests opened 2020-01-01 to 2026-07-31. Grain: one row = one closed request. No population denominator is used here.

**Measure:** `days_to_close = close_date - request_date` in whole days (dates are date-only UTC, so a same-day close is 0). A request is *slower than typical for its type* when it took longer than the citywide median for its exact `request_type`, computed on this analysis set. That is not the City's target: the escalation targets are not downloaded, so nothing here is "late" or "on target".

**Change from the plan:** the type control was planned as the 17 `request_type_group` values (ANALYSIS_STANDARDS section 3) and changed after the first run to the 155 exact `request_type` values, which match the task file's wording ("the median for their own request type"). The groups hide large timing differences inside them (finding 3). The group-level result is reported alongside in `type_corrected_share_by_district.csv`, `median_days_by_district_and_type_group.csv` and `stat_tests.csv`.

## Findings

1. {f1}
2. {f2}
3. {f3}

## Caveats

- Closed requests only ({share_closed:.1%} of the analysis set). Requests still open on the extract date are missing, so the latest months lean fast.{cens}
- {n_open_with_date:,} requests are On Hold, In Progress or Open but carry a close date; the Closed filter drops them.
- Whole-day dates: a one-day difference is within the rounding, and a late-evening request can land on the next day.
- {len(small_types)} request types have fewer than {SMALL_CELL} closed requests ({n_small_rows:,} requests, {n_small_rows / n_closed:.1%}); their "typical" time rests on few cases.
- Outliers are kept: {n_over_365:,} requests ({n_over_365 / n_closed:.2%}) took over 365 days; medians with and without them are in `outlier_check.csv`.
- With {n_closed:,} rows nearly every p-value is tiny, so the effect sizes lead. Effect-size labels are rules of thumb (epsilon squared: Tomczak & Tomczak 2014; Cramér's V: Cohen 1988).
- Within-type Kruskal-Wallis tests are exploratory and Holm-corrected (`stat_tests.csv`).
- These are associations, not causes. Nothing here says why a district is faster or slower, or why districts file different request types.
- {n_no_district:,} in-window requests with no district (were `N/A` or `NONE`) are excluded.

## Filter log

| Step | Rows after | Rows removed |
|---|---|---|
{flog_md}

`n_raw` = {n_raw:,} · `n_window` = {n_window:,} · `n_final` = {n_final:,} · `n_closed` = {n_closed:,}. Outside the window: {n_before:,} before 2020-01-01 and {n_after:,} after 2026-07-31.

## Outputs

| File | What it holds |
|---|---|
| `days_to_close_by_district.csv` | Per district: `n`, `mean_days`, `median_days`, `p25_days`, `p75_days`, `p90_days`, `max_days`, `share_same_day` (closed the day it was opened), `share_over_365` |
| `type_corrected_share_by_district.csv` | Per district: `n`, `n_slower_than_type` / `share_slower_than_type` (above the citywide median for its exact type; the main measure), `share_slower_than_type_group` (same with the 17 groups), `share_slower_than_city_median` (raw yardstick), `share_trash_recycle`, `pp_vs_citywide_type` (points above or below the citywide share) |
| `type_medians_citywide.csv` | Per exact type: its group, `n`, `median_days` (the "typical" yardstick) |
| `trash_recycle_mix_by_district.csv` | The 5 most common exact types inside Trash/Recycle Services: each district's `n` and `share_of_district_trash_recycle`, and `citywide_type_median_days` |
| `median_days_by_district_and_type.csv` | District x the {TOP_K} most common exact types: `n`, `median_days`, `mean_days`, `p90_days`, `citywide_median_days`, `small_cell` (n < {SMALL_CELL}) |
| `median_days_by_district_and_type_group.csv` | The same at the 17-group level, for comparison |
| `stat_tests.csv` | Every test: `plan`, `scope`, `n`, `statistic`, `df`, `p_value`, `p_holm`, `effect_size_name`, `effect_size` |
| `chi2_residuals_by_district.csv` | Observed vs expected slower-than-typical counts and the Pearson residual per district |
| `outlier_check.csv` | Requests over the IQR fence ({fence:g} days) and over 365 days, and medians and means with and without the over-365 requests |
| `filter_log.csv` | The filter log above |
| `type_corrected_share_by_district.png`, `raw_vs_type_corrected_by_district.png`, `heatmap_median_days_district_by_type.png` | Charts, 200 dpi, 12 x 6.75 in |
"""
assert "\u2014" not in findings, "no em dashes in repo writing"
(OUT / "findings.md").write_text(findings, encoding="utf-8")
print(findings)

## 6. For the group chat and the talk (not saved to the repo)

In [ ]:
print("=== Chat post ===")
print(f"Time-to-close by district ({WINDOW_TXT}, n = {n_closed:,}):")
print(show.rename(columns={"same_day": "same_day_%"}).to_string(index=False))
print(f"Type-corrected (exact request type): share slower than typical {100 * s_t[lo_t]:.1f}% ({nice(lo_t)}) to "
      f"{100 * s_t[hi_t]:.1f}% ({nice(hi_t)}). District gap {100 * spread_raw:.1f} pts raw -> "
      f"{100 * spread_type:.1f} pts once type is accounted for (17 broad groups only get it to "
      f"{100 * spread_group:.1f}). Cramér's V {V_raw:.3f} raw -> {V:.3f}.")
print("Anything odd:" + ("".join(f"\n- {o}" for o in odd) if odd else " nothing beyond the documented gotchas."))

mix_say = {"shrinks": f"– about {shrink:.0%} of the raw gap lines up with what each district asks for",
           "widens": "– so the gap is not just type mix",
           "similar": "– so type mix accounts for little of it"}[MIX]
print("\n=== Talking points, about a minute ===")
print(f"""1. Raw numbers first: the typical request closes in {days_txt(city['median_days'])}. {fast_txt} {'are' if ' and ' in fast_txt else 'is'} fastest at a median of
   {days_txt(med.min())} and {slow_txt} slowest at {days_txt(med.max())}.
2. But districts ask for different things. {short(TOP_TR)} requests take about {days_txt(type_med[TOP_TR])}, and they are
   {top_share[hi_mix]:.0%} of {nice(hi_mix)}'s trash requests but only {top_share[lo_mix]:.0%} of {nice(lo_mix)}'s.
3. So we compared every request with the typical time for its exact type. The gap between districts goes from
   {100 * spread_raw:.1f} points to {100 * spread_type:.1f} points {mix_say}. (Raw vs type-corrected chart.)
4. Same type, similar speed: {short(TOP_TR)} requests close in {top_med_d.min():g} to {days_txt(top_med_d.max())} in every district. (Heatmap.)
5. Limits: "typical" is the citywide median, not the City's target – those targets aren't downloaded yet,
   so we never say "late". These are associations. Next iteration: score against the City's escalation times.""")